In [1]:
import os

import pandas as pd
import matplotlib.pyplot as plt
from pyvis.network import Network
from bokeh.plotting import figure, from_networkx, output_file, save
from bokeh.models import Circle, MultiLine
from bokeh.io import output_file, save

import psycopg2
import networkx as nx

In [3]:
# Database connection
try:
    conn = psycopg2.connect(
        dbname=os.getenv("POSTGRES_DB", "postgres"),
        user=os.getenv("POSTGRES_USER", "postgres"),
        password=os.getenv("POSTGRES_PASSWORD", "1966"),
        host=os.getenv("POSTGRES_HOST", "postgres"),
        port=int(os.getenv("POSTGRES_PORT", "5432"))
    )
    
    # Test the connection
    with conn.cursor() as cur:
        cur.execute("SELECT version();")
        db_version = cur.fetchone()
        print("Connection successful.")
        print(f"PostgreSQL version: {db_version[0]}")
        
        # Check if competenze table exists
        cur.execute("""
            SELECT EXISTS (
                SELECT FROM information_schema.tables 
                WHERE table_name = 'competenze'
            );
        """)
        table_exists = cur.fetchone()[0]
        print(f"'competenze' table exists: {table_exists}")
    
    conn.close()
    print("Connection closed successfully.")
    
except psycopg2.Error as e:
    print(f"Database connection failed: {e}")
except Exception as e:
    print(f"Error: {e}")

Connection successful.
PostgreSQL version: PostgreSQL 16.12 on aarch64-unknown-linux-musl, compiled by gcc (Alpine 15.2.0) 15.2.0, 64-bit
'competenze' table exists: True
Connection closed successfully.


In [4]:
def build_graph_direct(conn):
    """Build bipartite graph from competenze table with colors and labels."""
    G = nx.Graph()
    
    with conn.cursor() as cur:
        # Query all edges directly from competenze table
        cur.execute("""
            SELECT DISTINCT id_profilo, id_competenza 
            FROM competenze 
            WHERE id_profilo IS NOT NULL AND id_competenza IS NOT NULL
        """)
        
        edges = cur.fetchall()
        
        # Add edges with prefixed node IDs to avoid collisions
        G.add_edges_from([
            (f"P_{id_profilo}", f"C_{id_competenza}") 
            for id_profilo, id_competenza in edges
        ])
        
        # Get profili with titles
        cur.execute("""
            SELECT DISTINCT c.id_profilo, p.titolo 
            FROM competenze c
            JOIN profili p ON c.id_profilo = p.id_profilo
            WHERE c.id_profilo IS NOT NULL
        """)
        profili_data = cur.fetchall()
        
        # Add profili node attributes
        for id_profilo, titolo in profili_data:
            node_id = f"P_{id_profilo}"
            if node_id in G.nodes:
                G.nodes[node_id]['type'] = 'profilo'
                G.nodes[node_id]['color'] = 'red'
                G.nodes[node_id]['label'] = titolo or f"Profilo {id_profilo}"
                G.nodes[node_id]['title'] = f"Profilo: {titolo}\nID: {id_profilo}"
        
        # Get competenze with titles
        cur.execute("""
            SELECT DISTINCT id_competenza, titolo 
            FROM competenze 
            WHERE id_competenza IS NOT NULL
        """)
        competenze_data = cur.fetchall()
        
        # Add competenze node attributes
        for id_competenza, titolo in competenze_data:
            node_id = f"C_{id_competenza}"
            if node_id in G.nodes:
                G.nodes[node_id]['type'] = 'competenza'
                G.nodes[node_id]['color'] = 'blue'
                G.nodes[node_id]['label'] = titolo or f"Competenza {id_competenza}"
                G.nodes[node_id]['title'] = f"Competenza: {titolo}\nID: {id_competenza}"
    
    return G

In [ ]:
# # Method 2: Pandas approach (easier for complex queries)
# def build_graph_pandas(conn):
#     query = """
#     SELECT id_profilo as source, id_competenza as target
#     FROM competenze 
#     WHERE id_profilo IS NOT NULL AND id_competenza IS NOT NULL
#     """
    
#     df = pd.read_sql_query(query, conn)
#     G = nx.from_pandas_edgelist(df, source='source', target='target')
    
#     # Add node types
#     profili_df = pd.read_sql_query("SELECT DISTINCT id_profilo as id FROM competenze WHERE id_profilo IS NOT NULL", conn)
#     for id in profili_df['id']:
#         G.nodes[id]['type'] = 'profilo'
        
#     comp_df = pd.read_sql_query("SELECT DISTINCT id_competenza as id FROM competenze WHERE id_competenza IS NOT NULL", conn)
#     for id in comp_df['id']:
#         G.nodes[id]['type'] = 'competenza'
    
#     return G

In [5]:
conn = psycopg2.connect(
    dbname=os.getenv("POSTGRES_DB", "postgres"),
    user=os.getenv("POSTGRES_USER", "postgres"),
    password=os.getenv("POSTGRES_PASSWORD", "1966"),
    host=os.getenv("POSTGRES_HOST", "postgres"),
    port=int(os.getenv("POSTGRES_PORT", "5432"))
)

# Build graph
G = build_graph_direct(conn)

# Close connection after use
conn.close()

# Inspect the graph
print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")
print(f"Profili nodes: {sum(1 for n in G.nodes() if G.nodes[n].get('type') == 'profilo')}")
print(f"Competenze nodes: {sum(1 for n in G.nodes() if G.nodes[n].get('type') == 'competenza')}")
print("Sample edges:", list(G.edges())[:5])

Nodes: 1392
Edges: 1111
Profili nodes: 281
Competenze nodes: 1111
Sample edges: [('P_108', 'C_166'), ('P_108', 'C_41'), ('P_108', 'C_40'), ('P_108', 'C_42'), ('P_234', 'C_1117')]


In [8]:
# Create graph with pyvis
net = Network(height='750px', width='100%', notebook=True)
net.from_nx(G)
net.save_graph('/app/profili_comp_graph.html')
print("Graph saved to /app/profili_comp_graph.html")

Graph saved to /app/profili_comp_graph.html
